<a href="https://colab.research.google.com/github/bangaru01/C_programing/blob/main/Sterimol_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# INSTALL — RUN ONCE IN GOOGLE COLAB
# ============================================================
!pip install -q morfeus-ml pandas numpy openpyxl


# ============================================================
# IMPORTS
# ============================================================
import os
import warnings
import pandas as pd
import numpy as np

from morfeus import Sterimol, read_xyz

warnings.filterwarnings("ignore")


# ============================================================
# GOOGLE DRIVE / XYZ FOLDER
# ============================================================
# Change this if your folder has a different name

xyz_folder = "/content/drive/MyDrive/xyz_files"


# ============================================================
# YOUR FIXED ATOM NUMBERING
# ============================================================

C2_ATOM = 18          # C2 carbon
H_AT_C2 = 43          # H attached to C2
R3_START_ATOM = 44    # First atom of R3


# ============================================================
# FIND ALL XYZ FILES
# ============================================================

xyz_files = sorted([
    f for f in os.listdir(xyz_folder)
    if f.lower().endswith(".xyz")
])

print(f"Found {len(xyz_files)} XYZ files.\n")


# ============================================================
# STORAGE
# ============================================================

results = []


# ============================================================
# PROCESS EVERY XYZ FILE
# ============================================================

for file in xyz_files:

    path = os.path.join(xyz_folder, file)

    try:

        # ----------------------------------------------------
        # READ XYZ
        # ----------------------------------------------------
        elements, coordinates = read_xyz(path)

        n_atoms = len(elements)


        # ----------------------------------------------------
        # CHECK ATOM NUMBERS
        # ----------------------------------------------------

        if n_atoms < R3_START_ATOM:
            raise ValueError(
                f"Only {n_atoms} atoms found. "
                f"R3 is supposed to start at atom {R3_START_ATOM}."
            )


        # ----------------------------------------------------
        # R3 = ATOMS 44 THROUGH LAST ATOM
        # ----------------------------------------------------

        R3_atoms = list(
            range(R3_START_ATOM, n_atoms + 1)
        )


        # ----------------------------------------------------
        # KEEP ONLY:
        #
        #   C2 = atom 18
        #   R3 = atoms 44 → last
        #
        # Everything else is excluded.
        # ----------------------------------------------------

        keep_atoms = set(
            [C2_ATOM] + R3_atoms
        )

        excluded_atoms = [
            i
            for i in range(1, n_atoms + 1)
            if i not in keep_atoms
        ]


        # ----------------------------------------------------
        # CHECK C2 → R3 DISTANCE
        #
        # Atom 18 = C2
        # Atom 44 = first R3 atom
        # ----------------------------------------------------

        c2_coord = np.array(
            coordinates[C2_ATOM - 1]
        )

        r3_coord = np.array(
            coordinates[R3_START_ATOM - 1]
        )

        c2_r3_distance = np.linalg.norm(
            c2_coord - r3_coord
        )


        # ----------------------------------------------------
        # STERIMOL CALCULATION
        #
        # dummy_index:
        #       C2 = 18
        #
        # attached_index:
        #       first R3 atom = 44
        # ----------------------------------------------------

        sterimol = Sterimol(
            elements,
            coordinates.copy(),
            dummy_index=C2_ATOM,
            attached_index=R3_START_ATOM,
            excluded_atoms=excluded_atoms,
            radii_type="crc",
            n_rot_vectors=3600
        )


        # ----------------------------------------------------
        # GET STERIMOL PARAMETERS
        # ----------------------------------------------------

        B1 = sterimol.B_1_value
        B5 = sterimol.B_5_value
        L = sterimol.L_value


        # ----------------------------------------------------
        # IDENTIFY R3 FROM FILE NAME
        #
        # If the filename is tBu_TS1.xyz, for example,
        # this will simply store the filename unless you
        # create a mapping below.
        # ----------------------------------------------------

        R3_name = os.path.splitext(file)[0]


        # ----------------------------------------------------
        # STORE RESULTS
        # ----------------------------------------------------

        results.append({

            "File": file,

            "R3": R3_name,

            "Number of atoms": n_atoms,

            "C2 atom": C2_ATOM,

            "H at C2": H_AT_C2,

            "R3 first atom": R3_START_ATOM,

            "R3 atom range":
                f"{R3_START_ATOM}-{n_atoms}",

            "C2-R3 distance (Å)":
                c2_r3_distance,

            "B1 (Å)": B1,

            "B5 (Å)": B5,

            "L (Å)": L

        })


        # ----------------------------------------------------
        # PRINT RESULT
        # ----------------------------------------------------

        print(
            f"{file:30s} "
            f"R3: {R3_name:15s} "
            f"C2-R3: {c2_r3_distance:.2f} Å   "
            f"B1: {B1:.2f} Å   "
            f"B5: {B5:.2f} Å   "
            f"L: {L:.2f} Å"
        )


    except Exception as e:

        print(f"ERROR in {file}: {e}")


# ============================================================
# CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(results)


# ============================================================
# SORT BY FILE NAME
# ============================================================

df = df.sort_values(
    by="File"
).reset_index(drop=True)


# ============================================================
# SAVE CSV
# ============================================================

csv_file = os.path.join(
    xyz_folder,
    "R3_Sterimol_results.csv"
)

df.to_csv(
    csv_file,
    index=False
)


# ============================================================
# SAVE EXCEL
# ============================================================

xlsx_file = os.path.join(
    xyz_folder,
    "R3_Sterimol_results.xlsx"
)

df.to_excel(
    xlsx_file,
    index=False
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("STERIMOL CALCULATION COMPLETED")
print("=" * 70)

print(f"Number of XYZ files processed: {len(df)}")
print(f"\nCSV saved at:")
print(csv_file)

print(f"\nExcel saved at:")
print(xlsx_file)

display(df)